# 03. Polars Exercise (30–40 min)

- **Work with a climate obstruction dataset** (German, no text) in a .ipynb with explanations, code, and exercises/questions.
- Please take your time to reflect and do the exercise, and look up documentation in **Polars** when needed.
- **Individually or in pairs**
- **~30–40 minutes + follow-up**

---

**Dataset:** `df_climate_obstruction_german_notext.csv`

**Goals:**
1. Load and query data with Polars (and see how it differs from Pandas)
2. Use **lazy evaluation** so large files are not fully read until needed
3. Select, filter, and aggregate with Polars’ **expression API**
4. Create derived columns and a simple join
5. Export a result

**Why Polars?** If you know **Pandas**, you’re used to loading the whole CSV into memory (`pd.read_csv`) and then filtering. **Polars** offers a different design: you build a **query plan** with `scan_csv` and only run it when you call `.collect()`. That helps with large files. Polars also drops the **index** (every column is just a column), uses **expressions** like `pl.col('x')` for filtering and new columns, and runs many operations in parallel. This exercise shows the same workflows in both libraries so you can compare.

## 0) Setup

In [1]:
# If needed:
# %pip install polars

import polars as pl
from pathlib import Path

data_path = Path('df_climate_obstruction_german_notext.csv')
print('File exists:', data_path.exists())

File exists: True


## Pandas vs Polars: same task, different syntax

Below we do the **same small task** in both libraries (load a sample, filter, select columns, show a few rows) so you can see the differences.

- **Pandas:** `read_csv` loads data into memory immediately (eager). You use column indexing `df['col']` and boolean masks `df[df['col'] == 1]`.
- **Polars:** `scan_csv` builds a **lazy** plan; nothing is read until `.collect()`. You use **expressions** like `pl.col('col')` and chain methods (`.filter()`, `.select()`). No index — every column is just a column.

In [2]:
# Use a small sample so we can run both libraries quickly (optional: increase or remove n_rows)
import pandas as pd
import time

SAMPLE_ROWS = 10_000

# --- Pandas (eager): loads SAMPLE_ROWS into memory immediately ---
start_time_pd = time.time()
df_pd = pd.read_csv(data_path, nrows=SAMPLE_ROWS)
result_pd = df_pd[df_pd['co_pred'] == 1][['actor_name', 'platform', 'domain_clean', 'co_pred']].head(5)
end_time_pd = time.time()
elapsed_pd = end_time_pd - start_time_pd

# --- Polars (lazy): builds a plan, runs it when we call .collect() ---
lf_pl = pl.scan_csv(data_path, n_rows=SAMPLE_ROWS)
start_time_pl = time.time()
result_pl = (
    lf_pl
    .filter(pl.col('co_pred') == 1)
    .select(['actor_name', 'platform', 'domain_clean', 'co_pred'])
    .head(5)
    .collect()
)
end_time_pl = time.time()
elapsed_pl = end_time_pl - start_time_pl

print("Pandas result (first 5 rows):")
display(result_pd)
print(f"\nPandas time: {elapsed_pd:.4f} seconds")

print("\nPolars result (first 5 rows):")
result_pl
print(f"\nPolars time: {elapsed_pl:.4f} seconds")

Pandas result (first 5 rows):


,actor_name,platform,domain_clean,co_pred
0,Bürger für Moosburg,facebook,NaN,1
1,Gegenwind Baden-Württemberg,facebook,facebook,1
2,AfD Fraktion Ribnitz-Damgarten,facebook,facebook,1
3,KeineReligionenKeineParteien,telegram,NaN,1
4,Mali Germany,vkontakte,journalistenwatch.com,1



Pandas time: 0.1351 seconds

Polars result (first 5 rows):

Polars time: 0.0691 seconds


## 1) Read Data (Lazy)

**Lazy evaluation:** `pl.scan_csv(path)` does **not** read the file yet. It returns a **LazyFrame** (a query plan). The plan is executed only when you call `.collect()`, and Polars can optimize the full pipeline (e.g. only read columns that are actually used). For large CSVs, that can save memory and time.

In [3]:
lf = pl.scan_csv(data_path)
print('Columns:', len(lf.collect_schema().names()))
print(lf.collect_schema().names())

Columns: 36
['message_id', 'actor_username', 'actor_name', 'co_pred', 'co_pred_proba', 'lang', 'platform', 'account_category', 'account_type', 'datetime', 'domain', 'followers', 'interactions', 'link', 'link_to_actor', 'post_url', 'title', 'xeno_pred', 'xeno_pred_proba', 'predicted_label', 'confidence_class_0', 'confidence_class_1', 'prediction_confidence', 'domain_clean', 'domain_category', 'confidence_score', 'domain_secondary_category', 'account_anno', 'actor_anno', 'emne_profil', 'climate_stance', 'level', 'ideology', 'party', 'actor_ica', 'n_posts']


**Query plan: make optimization concrete**

You can inspect *what* Polars will do before running it. `.explain()` on a LazyFrame prints the **query plan**. Read it **bottom to top**: you’ll see how filters and column selection are **pushed down** into the CSV scan (fewer columns read, rows filtered early), instead of loading everything and then filtering.

In [4]:
# Same kind of pipeline we use later: filter + select + limit
plan = (
    lf
    .filter(pl.col('co_pred') == 1)
    .select(['actor_name', 'platform', 'domain_clean', 'co_pred'])
    .head(10)
)
print(plan.explain())

simple π 4/4 ["actor_name", "platform", ... 2 other columns]
  SLICE[offset: 0, len: 10]
    Csv SCAN [df_climate_obstruction_german_notext.csv]
    PROJECT 4/36 COLUMNS
    SELECTION: [(col("co_pred")) == (1)]
    ESTIMATED ROWS: 421532


In [5]:
df = pl.read_csv(data_path, infer_schema =0)
df
df["xeno_pred"].value_counts(sort=True)

xeno_pred,count
str,u32
null,344408
"""0""",95846
"""1""",6468


In [6]:
df = pl.read_csv(data_path, infer_schema =0)
df

message_id,actor_username,actor_name,co_pred,co_pred_proba,lang,platform,account_category,account_type,datetime,domain,followers,interactions,link,link_to_actor,post_url,title,xeno_pred,xeno_pred_proba,predicted_label,confidence_class_0,confidence_class_1,prediction_confidence,domain_clean,domain_category,confidence_score,domain_secondary_category,account_anno,actor_anno,emne_profil,climate_stance,level,ideology,party,actor_ica,n_posts
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""625900200782477_37759620457762…","""625900200782477""","""Bürger für Moosburg""","""1""","""0.9746943712234497""","""de""","""facebook""","""none""","""facebook_group""","""2020-12-29T21:48:12.000000""","""po.neopresse.com""","""757.0""","""9""","""https://po.neopresse.com/labor…","""https://www.facebook.com/62590…","""https://www.facebook.com/group…","""altmed_germany""",null,null,"""1""","""0.17453636""","""0.8254636""","""0.8254636""",null,null,null,null,"""group""","""citizen""","""fr""","""ambi""","""l""",null,null,"""group, citizen, fr""","""589"""
"""209299992511579_30404446527304…","""209299992511579""","""Gegenwind Baden-Württemberg""","""1""","""0.97367924451828""","""de""","""facebook""","""none""","""facebook_group""","""2020-07-06T10:44:14.000000""","""facebook.com/205737623753""","""266.0""","""0""","""https://www.facebook.com/20573…","""https://www.facebook.com/group…","""https://www.facebook.com/group…","""altmed_denmark""",null,null,"""1""","""0.10288414""","""0.8971158""","""0.8971158""","""facebook""","""Social Media Platforms""",null,null,null,null,null,null,null,null,null,null,"""147"""
"""589510718207462_12332278771690…","""589510718207462""","""AfD Fraktion Ribnitz-Damgarten""","""1""","""0.9707966446876526""","""de""","""facebook""","""POLITICAL_ORGANIZATION""","""facebook_page""","""2021-11-10T07:15:53.000000""","""facebook.com/leiferikholm2""","""1123.0""","""2""","""https://www.facebook.com/leife…","""https://www.facebook.com/58951…","""https://www.facebook.com/58951…","""altmed_germany""","""0""","""0.9996456503868104""","""1""","""0.07064436""","""0.9293557""","""0.9293557""","""facebook""","""Social Media Platforms""",null,null,null,null,null,null,null,null,null,null,"""266"""
"""27153_1194048097""","""KeineReligionenKeineParteien""","""KeineReligionenKeineParteien""","""1""","""0.9515879154205322""","""de""","""telegram""","""telegram""","""telegram""","""2022-01-05T13:48:51.000000""",null,"""0.0""","""1""","""0""","""https://t.me/KeineReligionenKe…","""https://t.me/KeineReligionenKe…","""altmed_germany""",null,null,"""1""","""0.30104017""","""0.6989599""","""0.6989599""",null,null,null,null,null,null,null,null,null,null,null,null,"""83"""
"""434502990_726""","""maligermany""","""Mali Germany""","""1""","""0.9740811586380005""","""de""","""vkontakte""","""vkontakte""","""vkontakte""","""2019-03-18T10:37:54.000000""","""journalistenwatch.com""","""0.0""","""0""","""https://www.journalistenwatch.…","""www.vk.com/maligermany""","""https://vk.com/maligermany?w=w…","""altmed_germany""","""0""","""0.999729335308075""","""1""","""0.19081134""","""0.80918866""","""0.80918866""","""journalistenwatch.com""","""Hyperpartisan and Alternative …","""3""","""Blogs/Personal Websites""",null,null,null,null,null,null,null,null,"""72"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""15568_1337506129""","""MutIstAllesWasDuBrauchst""","""MutIstAllesWasDuBrauchst""","""1""","""0.9580383896827698""","""de""","""telegram""","""telegram""","""telegram""","""2020-11-23T06:41:21.000000""","""epochtimes.de""","""0.0""","""20""","""https://www.epochtimes.de/poli…","""https://t.me/MutIstAllesWasDuB…","""https://t.me/MutIstAllesWasDuB…","""altmed_germany""",null,null,"""1""","""0.17684413""","""0.8231558""","""0.8231558""","""epochtimes.de""","""Hyperpartisan and Alternative …","""3""","""Mainstream News Sites""",null,null,

## 2) First Look

**In Polars:** You **select** columns explicitly and **limit** rows with `.head(n)`. Only when you call `.collect()` is the lazy plan run and you get a small DataFrame.  
**Exercise:** Collect a small preview and inspect key columns: `actor_name`, `platform`, `domain_clean`, `co_pred`.

In [7]:
preview = lf.select(['actor_name','platform','domain_clean','co_pred']).head(10).collect()
preview

actor_name,platform,domain_clean,co_pred
str,str,str,i64
"""Bürger für Moosburg""","""facebook""",null,1
"""Gegenwind Baden-Württemberg""","""facebook""","""facebook""",1
"""AfD Fraktion Ribnitz-Damgarten""","""facebook""","""facebook""",1
"""KeineReligionenKeineParteien""","""telegram""",null,1
"""Mali Germany""","""vkontakte""","""journalistenwatch.com""",1
"""AfD Heidelberg""","""facebook""","""facebook""",1
"""faz_net""","""telegram""","""media0.faz.net""",1
"""koppreport""","""telegram""","""rnd.de""",1
"""Mehriel☦️""","""twitter""",null,1


## 3) Basic Filtering

**Expressions:** Conditions use `pl.col('column_name')` and methods like `.is_not_null()`. Combine with `&` and `|` (and wrap in parentheses). This is the **expression API** — the same style is used for new columns later.  
**Exercise:** Keep only rows where `co_pred == 1` and `domain_clean` is not null.

In [8]:
filtered = lf.filter((pl.col('co_pred') == 1) & (pl.col('domain_clean').is_not_null()))
filtered.select(['actor_name','domain_clean','co_pred']).head(10).collect()

actor_name,domain_clean,co_pred
str,str,i64
"""Gegenwind Baden-Württemberg""","""facebook""",1
"""AfD Fraktion Ribnitz-Damgarten""","""facebook""",1
"""Mali Germany""","""journalistenwatch.com""",1
"""AfD Heidelberg""","""facebook""",1
"""faz_net""","""media0.faz.net""",1
"""koppreport""","""rnd.de""",1
"""FREIE PRESSE""","""sciencefiles.org""",1
"""allesausser""","""bild.de""",1
"""AfD Heidelberg""","""epochtimes.de""",1


## 4) Group By: Top Domains

**Group + aggregate:** Like Pandas’ `groupby` + `agg`, but with `.group_by()` and `.agg()`. Use `pl.len()` for row count and `.alias('name')` to name the result. Chain `.sort()` and `.head()` to get top-N.  
**Exercise:** Count rows per `domain_clean` and show top 15 domains.

In [9]:
top_domains = (
    filtered
    .group_by('domain_clean')
    .agg(pl.len().alias('n_rows'))
    .sort('n_rows', descending=True)
    .head(15)
    .collect()
)
top_domains

domain_clean,n_rows
str,u32
"""facebook""",32151
"""eike-klima-energie.eu""",17470
"""welt.de""",12880
"""focus.de""",10735
"""tichyseinblick.de""",9994
…,…
"""telegram""",4341
"""reitschuster.de""",4266
"""spiegel.de""",4125


## 5) Group By: Top Actors

Same pattern as above: group by a column, aggregate (e.g. count), sort, take top 15.  
**Exercise:** Count rows per `actor_name` and show top 15 actors.

In [10]:
top_actors = (
    filtered
    .group_by('actor_name')
    .agg(pl.len().alias('n_rows'))
    .sort('n_rows', descending=True)
    .head(15)
    .collect()
)
top_actors

actor_name,n_rows
str,u32
"""Fakten gegen Klimahysterie""",3070
"""Europäisches Institut für Klim…",2969
"""Viktor Baranoff""",2736
"""Eddie Graf""",2665
"""tageszeitung""",2660
…,…
"""Gegenwind Deutschland""",1640
"""Weinheims konservative Mitte""",1501
"""Klima-Wahrheiten""",1466


## 6) Derived Column

**New columns:** `.with_columns()` adds one or more columns. Each column is an **expression**, e.g. `(pl.col('prediction_confidence') >= 0.9).alias('high_conf')`. You can then use the new column in `.group_by()` and `.agg()` (e.g. `.mean()` for share).  
**Exercise:** Create a `high_conf` indicator where `prediction_confidence >= 0.9`, then summarize share of high confidence by platform.

In [11]:
platform_conf = (
    filtered
    .with_columns((pl.col('prediction_confidence') >= 0.9).alias('high_conf'))
    .group_by('platform')
    .agg([
        pl.len().alias('n_rows'),
        pl.col('high_conf').mean().alias('share_high_conf')
    ])
    .sort('n_rows', descending=True)
    .collect()
)
platform_conf


platform,n_rows,share_high_conf
str,u32,f64
"""facebook""",128974,0.272946
"""telegram""",78712,0.229838
"""twitter""",35739,0.341364
"""vkontakte""",30729,0.234241
"""gab""",3620,0.18232
"""fourchan""",20,0.0
"""reddit""",16,0.125


## 7) Mini Join Exercise

**Joins:** Polars uses `.join(other, on='col', how='left')`. Here we attach each top actor’s most common `account_type` by first computing that per actor, then joining to the top-15 table.  
**Exercise:** Join top actors with their most common `account_type`.

In [12]:
actor_type = (
    filtered
    .group_by(['actor_name', 'account_type'])
    .agg(pl.len().alias('n'))
    .sort(['actor_name', 'n'], descending=[False, True])
    .group_by('actor_name')
    .agg(pl.col('account_type').first().alias('top_account_type'))
)

top_actor_with_type = top_actors.join(actor_type.collect(), on='actor_name', how='left')
top_actor_with_type

actor_name,n_rows,top_account_type
str,u32,str
"""Fakten gegen Klimahysterie""",3070,"""facebook_group"""
"""Europäisches Institut für Klim…",2969,"""facebook_page"""
"""Viktor Baranoff""",2736,"""twitter_account"""
"""Eddie Graf""",2665,"""twitter_account"""
"""tageszeitung""",2660,"""telegram"""
…,…,…
"""Gegenwind Deutschland""",1640,"""facebook_group"""
"""Weinheims konservative Mitte""",1501,"""facebook_group"""
"""Klima-Wahrheiten""",1466,"""facebook_group"""


## Polars ↔ Pandas: converting back and forth

You can convert between the two so you can use the right tool for each step:

- **Polars → Pandas:** `df_pandas = pl_df.to_pandas()` — use when you need a Pandas-only library (e.g. some stats or plotting APIs), or when sharing with code that expects a Pandas DataFrame.
- **Pandas → Polars:** `pl_df = pl.from_pandas(df_pandas)` — use when you’ve loaded or received data in Pandas and want to run Polars’ lazy pipeline or expression API on it.

**When this is useful:** Do heavy I/O and filtering in Polars (lazy, fast), then convert to Pandas for a small result if you need `.plot()`, scikit-learn, or other Pandas-based tools. Or start from Pandas data (e.g. from an API) and convert to Polars for complex transformations before converting back for downstream use.

**Note:** Polars uses **PyArrow** for `.to_pandas()`. If you get `ModuleNotFoundError: No module named 'pyarrow'`, run the cell below once, then **restart the kernel** (e.g. Run → Restart) and re-run from here. 

In [ ]:
# %pip install pyarrow

In [13]:
# Example: we already have top_domains (Polars). Convert to Pandas and back.
import pandas as pd

as_pandas = top_domains.to_pandas()
print("Polars → Pandas:", type(as_pandas), as_pandas.shape)

back_to_polars = pl.from_pandas(as_pandas)
print("Pandas → Polars:", type(back_to_polars), back_to_polars.shape)
back_to_polars.head(3)

Polars → Pandas: <class 'pandas.core.frame.DataFrame'> (15, 2)
Pandas → Polars: <class 'polars.dataframe.frame.DataFrame'> (15, 2)


domain_clean,n_rows
str,u32
"""facebook""",32151
"""eike-klima-energie.eu""",17470
"""welt.de""",12880


## 8) Export Result

In [14]:
out = Path('03_polars_top_domains.csv')
top_domains.write_csv(out)
print('Saved:', out)

Saved: 03_polars_top_domains.csv


## 9) Reflection Questions

1. **Lazy vs eager:** Why might `scan_csv` + lazy queries be better than `pd.read_csv` for a large CSV? When would you still use Pandas?
2. **Syntax:** In your own words, what does `group_by(...).agg(...)` do? How does that compare to Pandas’ `groupby` + `agg`?
3. **Filter vs group:** What changes if we treat `co_pred` as a filter condition (only keep rows where `co_pred == 1`) versus a grouping variable (group by `co_pred` and count)?
4. **Usefulness:** For this climate-obstruction dataset, which Polars features (lazy read, expressions, no index) did you find most helpful? Would you use Polars for a similar project?